# Блокнот 16. Детекция: рамки, IoU, NMS, AP и Faster R-CNN

Курс «Машинное обучение с нуля», модуль 16. Разделы: 1 — урок 16.1, 2 — урок 16.2, 3 — урок 16.3, 4 — уроки 16.4–16.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Раздел 4 скачивает одну картинку из набора COCO (≈170 КБ) и веса Faster R-CNN (≈75 МБ на процессоре, ≈170 МБ с видеокартой).

In [ ]:
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from matplotlib.patches import Rectangle
from PIL import Image
from torchvision.ops import box_iou, nms

device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")
os.makedirs(DATA, exist_ok=True)
print("устройство:", device, "| torchvision", torchvision.__version__)

## 1. Форматы рамок (урок 16.1)

**Задание 1.** Допишите переводы в формат xyxy: из COCO (`x, y, w, h`) и из YOLO (`cx, cy, w, h` долями от размера картинки `W × H`).

In [ ]:
def xywh_to_xyxy(b):
    x, y, w, h = b
    ...  # ваш код


def yolo_to_xyxy(b, W, H):
    cx, cy, w, h = b
    ...  # ваш код


assert np.allclose(xywh_to_xyxy((160, 120, 240, 240)), (160, 120, 400, 360))
assert np.allclose(yolo_to_xyxy((0.4375, 0.5, 0.375, 0.5), 640, 480), (160, 120, 400, 360))


def draw(ax, boxes, labels=None, color="lime"):
    for k, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=2))
        if labels is not None:
            ax.text(x1, y1 - 3, labels[k], color="white", fontsize=8, bbox=dict(fc=color, ec="none", pad=1))


canvas = np.full((480, 640, 3), 0.92)
fig, ax = plt.subplots(figsize=(5, 3.8))
ax.imshow(canvas)
draw(ax, [yolo_to_xyxy((0.4375, 0.5, 0.375, 0.5), 640, 480)], ["из YOLO"], "tab:green")
draw(ax, [xywh_to_xyxy((400, 50, 150, 100))], ["из COCO"], "tab:blue")
ax.set_title("Нарисуйте рамки, прежде чем обучать!")
plt.show()

## 2. IoU и NMS своими руками (урок 16.2)

**Задание 2.** Допишите `iou_matrix` — IoU каждой рамки из `A` с каждой из `B` (векторно, без циклов) — и `my_nms`. Проверка — сравнение с `torchvision.ops`.

In [ ]:
def iou_matrix(A, B):
    """A: (n, 4), B: (m, 4) в формате xyxy → матрица (n, m)."""
    ...  # ваш код: транслирование (n, 1) против (1, m)


def my_nms(boxes, scores, thr):
    order = np.argsort(-scores)
    keep = []
    ...  # ваш код
    return keep


rng = np.random.default_rng(0)
xy = rng.uniform(0, 500, size=(60, 2))
wh = rng.uniform(20, 120, size=(60, 2))
boxes = np.hstack([xy, xy + wh])
scores = rng.uniform(size=60)
assert np.allclose(iou_matrix(boxes, boxes), box_iou(torch.tensor(boxes), torch.tensor(boxes)).numpy())
mine = my_nms(boxes, scores, 0.3)
ref = nms(torch.tensor(boxes), torch.tensor(scores), 0.3).tolist()
print("свой NMS оставил", len(mine), "рамок из 60, совпадает с torchvision:", mine == ref)
assert mine == ref

## 3. Сопоставление и AP (урок 16.3)

**Задание 3.** Допишите сопоставление предсказаний одного класса с настоящими рамками: идти по убыванию уверенности; предсказание верное, если его лучшая IoU с ещё **не занятой** настоящей рамкой не меньше порога, — тогда эта рамка занимается.

In [ ]:
def match(pred, scores, gt, thr=0.5):
    order = np.argsort(-scores)
    used = np.zeros(len(gt), bool)
    tp = []
    ious = iou_matrix(pred, gt) if len(gt) else np.zeros((len(pred), 0))
    for i in order:
        ...  # ваш код: добавьте в tp 1 или 0
    return tp


def average_precision(tp, n_gt):
    tp = np.array(tp, float)
    cum = np.cumsum(tp)
    r = np.concatenate([[0], cum / n_gt, [1]])
    p = np.concatenate([[0], cum / np.arange(1, len(tp) + 1), [0]])
    for i in range(len(p) - 2, -1, -1):
        p[i] = max(p[i], p[i + 1])
    return float(np.sum((r[1:] - r[:-1]) * p[1:]))


gt = np.array([[10, 10, 50, 50], [100, 100, 160, 150], [200, 30, 260, 90]], float)
pred = np.array([[12, 8, 52, 49], [11, 12, 49, 51], [98, 105, 158, 152], [300, 300, 340, 340], [205, 40, 250, 95]], float)
sc = np.array([0.95, 0.9, 0.8, 0.7, 0.6])
tp = match(pred, sc, gt)
print("верные по порядку уверенности:", tp)
assert tp == [1, 0, 1, 0, 1]
for thr in (0.5, 0.75):
    print(f"AP при пороге IoU {thr}: {average_precision(match(pred, sc, gt, thr), len(gt)):.3f}")

## 4. Готовый Faster R-CNN из torchvision (уроки 16.4–16.5)

Скачаем картинку из набора COCO (две кошки на диване — классический пример из документации) и обученный на COCO детектор. На процессоре берём лёгкую основу MobileNetV3, с видеокартой — ResNet-50 FPN v2.

In [ ]:
img_path = os.path.join(DATA, "coco_000000039769.jpg")
if not os.path.exists(img_path):
    urllib.request.urlretrieve("http://images.cocodataset.org/val2017/000000039769.jpg", img_path)
img = Image.open(img_path).convert("RGB")

if device == "cuda":
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights=weights)
else:
    weights = torchvision.models.detection.FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights=weights)
model = model.eval().to(device)
x = weights.transforms()(img).to(device)
with torch.no_grad():
    t0 = time.time()
    out = model([x])[0]
print(f"время: {(time.time() - t0) * 1000:.0f} мс, рамок до порога уверенности: {len(out['boxes'])}")

names = weights.meta["categories"]
keep = out["scores"] > 0.5
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img)
draw(ax, out["boxes"][keep].cpu().numpy(),
     [f"{names[l]} {s:.2f}" for l, s in zip(out["labels"][keep].tolist(), out["scores"][keep].tolist())], "tab:orange")
ax.axis("off")
plt.show()
found = [names[l] for l in out["labels"][keep].tolist()]
print("найдено:", found)
assert "cat" in found

Заглянем внутрь: сколько якорей в каждой точке и сколько областей RPN отдаёт второй ступени.

In [ ]:
print("якорей в каждой точке каждого уровня:", model.rpn.anchor_generator.num_anchors_per_location())
print("областей после RPN при применении:", model.rpn.post_nms_top_n())
print("порог NMS итоговых рамок:", model.roi_heads.nms_thresh)
print("параметров:", sum(p.numel() for p in model.parameters()))

Настройки готовой модели отличаются от статьи 2015 года: в torchvision у этого варианта 15 якорей в каждой точке (5 размеров × 3 соотношения сторон) и 1000 областей на вторую ступень вместо 9 и 300. Принцип тот же — меняются только числа, которые подбирают под набор данных и скорость.